In [1]:
import sys, os
sys.path.insert(0, '/usr/local/spark/python')
import pyspark
print('PySpark', pyspark.__version__, '| Python', sys.executable)
print('IMDb files:', os.listdir('/home/jovyan/work/imdb'))

PySpark 3.3.1 | Python /opt/conda/bin/python
IMDb files: ['name.basics.tsv.gz', 'title.basics.tsv.gz', 'title.ratings.tsv.gz']


In [2]:
import sys
sys.path.insert(0, "/usr/local/spark/python")

from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("Practico IMDb")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

DATA_DIR = "/home/jovyan/work/imdb"


def read_tsv(filename, schema):
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("nullValue", r"\N")
        .schema(schema)
        .csv(f"{DATA_DIR}/{filename}")
    )


# Esquemas de IMDb: identificadores y textos como string; años, minutos y votos como enteros.
people_schema = T.StructType([
    T.StructField("nconst", T.StringType(), nullable=False),
    T.StructField("primaryName", T.StringType()),
    T.StructField("birthYear", T.IntegerType()),
    T.StructField("deathYear", T.IntegerType()),
    T.StructField("primaryProfession", T.StringType()),
    T.StructField("knownForTitles", T.StringType()),
])

titles_schema = T.StructType([
    T.StructField("tconst", T.StringType(), nullable=False),
    T.StructField("titleType", T.StringType()),
    T.StructField("primaryTitle", T.StringType()),
    T.StructField("originalTitle", T.StringType()),
    T.StructField("isAdult", T.IntegerType()),
    T.StructField("startYear", T.IntegerType()),
    T.StructField("endYear", T.IntegerType()),
    T.StructField("runtimeMinutes", T.IntegerType()),
    T.StructField("genres", T.StringType()),
])

ratings_schema = T.StructType([
    T.StructField("tconst", T.StringType(), nullable=False),
    T.StructField("averageRating", T.DoubleType()),
    T.StructField("numVotes", T.IntegerType()),
])

people = read_tsv("name.basics.tsv.gz", people_schema)
titles = read_tsv("title.basics.tsv.gz", titles_schema)
ratings = read_tsv("title.ratings.tsv.gz", ratings_schema)


# Exploración de name.basics: muestra, filtro por profesión y revisión de nulos.
people.select("nconst", "primaryName", "primaryProfession").show(3, truncate=False)

people.filter(
    F.col("primaryProfession").rlike(r"(^|,)(actor|actress)(,|$)")
).select(
    "primaryName", "birthYear", "primaryProfession"
).limit(3).show(truncate=False)

people.agg(
    F.count("*").alias("personas"),
    F.sum(F.col("birthYear").isNull().cast("int")).alias("sin_anio_nacimiento"),
    F.sum(F.col("primaryProfession").isNull().cast("int")).alias("sin_profesion"),
).show()


# Exploración de title.basics: muestra, películas recientes y frecuencia por tipo.
titles.select(
    "tconst", "titleType", "primaryTitle", "startYear", "genres"
).show(3, truncate=False)

titles.filter(
    (F.col("titleType") == "movie") & (F.col("startYear") >= 2020)
).select(
    "primaryTitle", "startYear", "genres"
).limit(3).show(truncate=False)

titles.groupBy("titleType").count().orderBy(
    F.desc("count")
).show(8, truncate=False)


# Exploración de title.ratings: muestra, títulos muy votados y tramos de rating.
ratings.select("tconst", "averageRating", "numVotes").show(3, truncate=False)

ratings.filter(F.col("numVotes") >= 100000).select(
    "tconst", "averageRating", "numVotes"
).orderBy(
    F.desc("averageRating")
).limit(5).show(truncate=False)

ratings.groupBy(
    F.floor("averageRating").cast("int").alias("tramo")
).count().orderBy("tramo").show(truncate=False)


# Medida integrada: rating ponderado por votos de películas conocidas del actor Morgan Freeman (nm0000151).
morgan_titles = (
    people.filter(
        (F.col("nconst") == "nm0000151")
        & F.col("primaryProfession").rlike(r"(^|,)(actor|actress)(,|$)")
    )
    .select(
        F.explode(
            F.split(F.coalesce(F.col("knownForTitles"), F.lit("")), ",")
        ).alias("tconst")
    )
    .filter(F.length("tconst") > 0)
    .dropDuplicates(["tconst"])
)

integrated = (
    F.broadcast(morgan_titles)
    .join(titles, on="tconst", how="inner")
    .join(ratings, on="tconst", how="inner")
    .filter(F.col("titleType") == "movie")
)

integrated.select(
    "primaryTitle", "startYear", "averageRating", "numVotes"
).orderBy(
    F.desc("numVotes")
).show(10, truncate=False)

weighted_rating = F.sum(
    F.col("averageRating") * F.col("numVotes")
) / F.sum("numVotes")

measure = integrated.agg(
    F.countDistinct("tconst").alias("peliculas"),
    F.sum("numVotes").alias("votos"),
    F.round(weighted_rating, 2).alias("rating_ponderado"),
)
measure.show(truncate=False)


+---------+---------------+---------------------------------+
|nconst   |primaryName    |primaryProfession                |
+---------+---------------+---------------------------------+
|nm0000001|Fred Astaire   |actor,miscellaneous,producer     |
|nm0000002|Lauren Bacall  |actress,miscellaneous,soundtrack |
|nm0000003|Brigitte Bardot|actress,music_department,producer|
+---------+---------------+---------------------------------+
only showing top 3 rows

+---------------+---------+---------------------------------+
|primaryName    |birthYear|primaryProfession                |
+---------------+---------+---------------------------------+
|Fred Astaire   |1899     |actor,miscellaneous,producer     |
|Lauren Bacall  |1924     |actress,miscellaneous,soundtrack |
|Brigitte Bardot|1934     |actress,music_department,producer|
+---------------+---------+---------------------------------+

+--------+-------------------+-------------+
|personas|sin_anio_nacimiento|sin_profesion|
+--------+------

In [3]:
# Medida integrada: comparación entre todas las personas con el nombre y un actor identificado por nconst.
# En ambos casos se siguen los knownForTitles y se integran títulos y ratings mediante join.

def known_titles_for(person_filter):
    return (
        people.filter(person_filter)
        .select(F.explode(F.split(F.coalesce(F.col("knownForTitles"), F.lit("")), ",")).alias("tconst"))
        .filter(F.length("tconst") > 0)
        .dropDuplicates(["tconst"])
    )

def integrate_titles(person_filter):
    return (
        F.broadcast(known_titles_for(person_filter))
        .join(titles, on="tconst", how="inner")
        .join(ratings, on="tconst", how="inner")
        .filter(F.col("titleType") == "movie")
    )

def show_case(label, person_filter):
    result = integrate_titles(person_filter)
    print(f"=== {label} ===")
    result.select("primaryTitle", "startYear", "averageRating", "numVotes").orderBy(
        F.desc("numVotes")
    ).show(10, truncate=False)
    weighted_rating = F.sum(F.col("averageRating") * F.col("numVotes")) / F.sum("numVotes")
    result.agg(
        F.countDistinct("tconst").alias("peliculas"),
        F.sum("numVotes").alias("votos"),
        F.round(weighted_rating, 2).alias("rating_ponderado"),
    ).show(truncate=False)

# Caso A: todas las personas identificadas como actor/actriz cuyo nombre exacto coincide.
name_filter = (
    (F.col("primaryName") == "Morgan Freeman")
    & F.col("primaryProfession").rlike(r"(^|,)(actor|actress)(,|$)")
)
people.filter(name_filter).select("nconst", "primaryName", "primaryProfession").show(truncate=False)
show_case("CASO A - nombre exacto Morgan Freeman (todas las coincidencias)", name_filter)

# Caso B: una sola persona, seleccionada con el identificador estable de IMDb.
id_filter = (
    (F.col("nconst") == "nm0000151")
    & F.col("primaryProfession").rlike(r"(^|,)(actor|actress)(,|$)")
)
show_case("CASO B - actor nm0000151", id_filter)


+---------+--------------+-----------------------+
|nconst   |primaryName   |primaryProfession      |
+---------+--------------+-----------------------+
|nm0000151|Morgan Freeman|actor,producer,director|
|nm0293532|Morgan Freeman|actor                  |
|nm6783391|Morgan Freeman|actor                  |
+---------+--------------+-----------------------+

=== CASO A - nombre exacto Morgan Freeman (todas las coincidencias) ===
+------------------------+---------+-------------+--------+
|primaryTitle            |startYear|averageRating|numVotes|
+------------------------+---------+-------------+--------+
|The Shawshank Redemption|1994     |9.3          |3265691 |
|Seven                   |1995     |8.6          |2078294 |
|Million Dollar Baby     |2004     |8.1          |777316  |
|Driving Miss Daisy      |1989     |7.3          |125264  |
|54                      |1998     |5.9          |36338   |
|The Big Hit             |1998     |6.1          |33583   |
|The Life Before This    |1999